# 3.11 · 不平衡数据 / Imbalanced Data

> **课程定位 / Where this fits**
> 第 11 课，**Part 3 · EDA 与数据预处理**。
> Lesson 11, **Part 3 · EDA & Preprocessing**.
>
> 欺诈、罕见病、点击率等场景里，正类常常只占 1% 甚至更少。这时**准确率会骗人**（全猜多数类就 99% 准），常规模型也会偏向多数类。这一课从预处理角度系统讲不平衡的应对：指标、类权重、重采样(SMOTE)、阈值移动。Part 5.14 会从分类器角度再深入。
> In fraud, rare disease, click-through, the positive class is often ≤1%. Here **accuracy lies** (predict the majority for 99%) and models drift toward the majority. This lesson tackles imbalance from the preprocessing side: metrics, class weights, resampling (SMOTE), threshold shifting. Part 5.14 goes deeper from the classifier side.
>
> 💼 **实战/面试视角**："数据不平衡怎么办 / SMOTE 原理 / 为什么不能在 CV 前 SMOTE" 几乎必问。
> 💼 **Practical/interview angle:** "how to handle imbalance / SMOTE / why not SMOTE before CV" are near-guaranteed.

> 💡 **面试相关 / Interview-relevant**
> - "不平衡为什么准确率失效 / 看什么指标"（出镜率 ★★★★★，PR-AUC/recall）
> - "class_weight 怎么起作用"（★★★★★）
> - "过采样 vs 欠采样 / SMOTE 原理"（★★★★★）
> - "为什么 SMOTE 必须在 CV 内做（防泄漏）"（★★★★★）
> - "阈值移动 vs 重采样"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解**准确率悖论**，知道不平衡看 **PR-AUC / recall**。
   Understand the accuracy paradox; watch PR-AUC / recall under imbalance.
2. 用 **class_weight='balanced'** 做代价敏感学习。
   Use class_weight='balanced' for cost-sensitive learning.
3. 掌握**欠采样/过采样/SMOTE**及其取舍。
   Master under/over-sampling and SMOTE.
4. 理解**为什么 SMOTE 必须在 CV 内**（imblearn Pipeline）。
   Understand why SMOTE must live inside CV (imblearn Pipeline).
5. 用**阈值移动**零成本换 recall，并理性看待各策略。
   Use threshold shifting to trade for recall at zero cost, and judge strategies soberly.

## 目录 / TOC
1. [先建直觉：准确率悖论 ⭐](#1)
2. [朴素模型的问题](#2)
3. [class_weight：代价敏感 ⭐](#3)
4. [重采样：欠采样/过采样/SMOTE ⭐](#4)
5. [SMOTE 的泄漏陷阱 ⭐](#5)
6. [阈值移动 ⭐](#6)
7. [横向对比 + 小结](#7)


<a id="1"></a>
## 1. 先建直觉：准确率悖论 ⭐ / Intuition: The Accuracy Paradox

如果 99% 的样本是"正常"、1% 是"欺诈"，那么一个**永远预测"正常"**的废模型就有 **99% 准确率**——但它一个欺诈都没抓到（recall = 0），毫无用处。这就是**准确率悖论**：不平衡时准确率被多数类主导，完全失去意义。
If 99% of samples are "normal" and 1% "fraud", a useless model that **always predicts "normal"** scores **99% accuracy** — yet catches zero fraud (recall = 0). This is the **accuracy paradox**: under imbalance, accuracy is dominated by the majority and becomes meaningless.

所以不平衡问题的核心是换个思路：**换指标**（看 recall / precision / PR-AUC，5.1 讲过）+ **让模型重视少数类**（权重/重采样/阈值）。
So the key shift: **change the metric** (recall / precision / PR-AUC, see 5.1) + **make the model care about the minority** (weights / resampling / thresholds).


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, f1_score, precision_score, recall_score,
                             average_precision_score)
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# 极端不平衡: 1% 正类(欺诈) / extreme 1% imbalance
X, y = make_classification(n_samples=10000, n_features=15, n_informative=6,
                           weights=[0.99, 0.01], flip_y=0.01, random_state=0)
print(f"类别分布 class counts: {np.bincount(y)} → 欺诈占 {y.mean():.1%}")

# 傻瓜模型: 永远猜多数类 → 高准确率但 0 召回 / dummy: always majority
dummy = DummyClassifier(strategy="most_frequent").fit(X, y)
print(f"\n傻瓜模型(永远猜非欺诈): 准确率 = {dummy.score(X, y):.1%}")
print(f"  但 recall(欺诈) = {recall_score(y, dummy.predict(X), zero_division=0):.1%} — 一个都没抓到!")
print("  → 99% 准确率 + 0% 召回 = 完全无用 = 准确率悖论")


<a id="2"></a>
## 2. 朴素模型的问题 / The Vanilla Model's Problem

直接训练一个普通逻辑回归（不做任何不平衡处理），准确率看着很高，但 **recall 很低**——它漏掉了大量欺诈。看混淆矩阵就一目了然。
Training a plain logistic regression (no imbalance handling) gives high accuracy but **low recall** — it misses lots of fraud. The confusion matrix makes it obvious.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
y_pred = clf.predict(X_te)

print("朴素逻辑回归 vanilla:")
print(f"  准确率 accuracy = {clf.score(X_te, y_te):.1%}  (看着不错)")
print(f"  precision = {precision_score(y_te, y_pred):.1%}")
print(f"  recall    = {recall_score(y_te, y_pred):.1%}  ← 召回低! 漏掉大量欺诈")
print(f"  F1        = {f1_score(y_te, y_pred):.1%}")
print(f"  PR-AUC    = {average_precision_score(y_te, clf.predict_proba(X_te)[:,1]):.3f}")
print("\n混淆矩阵(行=真实, 列=预测) confusion matrix:")
print(confusion_matrix(y_te, y_pred))
print("→ 模型偏向多数类, 召回不足, 需要专门处理")


<a id="3"></a>
## 3. class_weight：代价敏感 ⭐ / Cost-Sensitive Learning

**最省事的方案**：不动数据，只在损失函数里**给少数类的错误更高的权重**。`class_weight='balanced'` 自动按类频率的倒数加权——等价于"少数类的每个样本算很多次"。无需重采样、无泄漏风险，是处理不平衡的首选起点。
**The easiest fix:** don't touch the data; just **weight the minority class's errors more heavily** in the loss. `class_weight='balanced'` auto-weights by inverse class frequency — like "each minority sample counts many times". No resampling, no leakage risk — the preferred starting point.


In [ ]:
clf_w = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
y_pred_w = clf_w.predict(X_te)
print("逻辑回归 + class_weight='balanced':")
print(f"  准确率 = {clf_w.score(X_te, y_te):.1%}  (略降 — 正常)")
print(f"  recall    = {recall_score(y_te, y_pred_w):.1%}  ← 大幅提升! 抓到更多欺诈")
print(f"  precision = {precision_score(y_te, y_pred_w):.1%}  (降了 — 误报变多, 这是权衡)")
print(f"  F1        = {f1_score(y_te, y_pred_w):.1%}")
# balanced 给每类的权重 = n_samples / (n_classes × 该类样本数)
print(f"\n自动权重: 非欺诈≈{len(y_tr)/(2*np.bincount(y_tr)[0]):.2f}, 欺诈≈{len(y_tr)/(2*np.bincount(y_tr)[1]):.1f}")
print("→ recall 大涨, 代价是 precision 下降(更多误报) — 这是不平衡处理的本质权衡")


<a id="4"></a>
## 4. 重采样：欠采样/过采样/SMOTE ⭐ / Resampling

另一条路是**改变数据的类别比例**（用 `imbalanced-learn`）：
The other route is to **change the class ratio of the data** (with `imbalanced-learn`):
- **随机欠采样**：丢掉部分多数类样本。快，但**丢信息**。
  **Random undersampling:** drop majority samples. Fast but **loses information**.
- **随机过采样**：复制少数类样本。不丢信息，但**易过拟合**（重复点）。
  **Random oversampling:** duplicate minority samples. No info loss but **overfits** (repeats).
- **SMOTE**（合成少数类过采样）：在少数类样本与其近邻之间**插值合成新点**，而非简单复制。更平滑，最常用。
  **SMOTE:** **synthesizes new points by interpolating** between a minority sample and its neighbors, not just copying. Smoother, most popular.


In [ ]:
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

print(f"原始 train: {np.bincount(y_tr)}")
for name, sampler in [("随机欠采样 under", RandomUnderSampler(random_state=0)),
                      ("随机过采样 over", RandomOverSampler(random_state=0)),
                      ("SMOTE", SMOTE(random_state=0))]:
    Xr, yr = sampler.fit_resample(X_tr, y_tr)   # fit_resample 返回重采样后的数据
    print(f"{name}: {np.bincount(yr)}")

X_sm, y_sm = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
clf_sm = LogisticRegression(max_iter=1000).fit(X_sm, y_sm)
print(f"\nSMOTE + 逻辑回归: recall={recall_score(y_te, clf_sm.predict(X_te)):.1%}, F1={f1_score(y_te, clf_sm.predict(X_te)):.1%}")


<a id="5"></a>
## 5. SMOTE 的泄漏陷阱 ⭐ / SMOTE's Leakage Trap

**这是面试高频陷阱**：SMOTE **必须在交叉验证的每一折内部、只对训练折做**。如果先对整个训练集 SMOTE 再做 CV，合成出来的少数类样本会有一部分"泄漏"进验证折（它们是用验证折的真实样本插值出来的近亲），评估虚高。
**A frequently-tested trap:** SMOTE **must be applied inside each CV fold, on the training part only.** If you SMOTE the whole training set before CV, some synthetic minority samples leak into the validation fold (they're near-clones interpolated from validation-fold reals), inflating the score.

正确做法：用 **`imblearn.pipeline.Pipeline`**（不是 sklearn 的），它会在 CV 时只在每折的训练部分 SMOTE，验证折永远是真实样本。
The fix: use **`imblearn.pipeline.Pipeline`** (not sklearn's), which applies SMOTE only to each fold's training part, keeping validation folds purely real.


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline   # 注意: imblearn 的 Pipeline

# ❌ 错: 先对全 train SMOTE, 再 CV → 合成样本泄漏进验证折 / WRONG: SMOTE before CV
X_sm_all, y_sm_all = SMOTE(random_state=0).fit_resample(X_tr, y_tr)
f1_wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sm_all, y_sm_all,
                           cv=StratifiedKFold(5), scoring="f1").mean()

# ✅ 对: SMOTE 放进 imblearn pipeline → CV 时每折只在 train 部分合成 / RIGHT: SMOTE inside pipeline
pipe = ImbPipeline([("smote", SMOTE(random_state=0)), ("clf", LogisticRegression(max_iter=1000))])
f1_right = cross_val_score(pipe, X_tr, y_tr, cv=StratifiedKFold(5), scoring="f1").mean()

print(f"❌ 先 SMOTE 再 CV:        F1 = {f1_wrong:.3f}  ← 虚高(合成样本泄漏进验证折)")
print(f"✅ SMOTE 在 pipeline 内:  F1 = {f1_right:.3f}  ← 诚实")
print("用 imblearn.pipeline.Pipeline(不是 sklearn 的); test 折永远是真实样本, 从不被合成污染")


<a id="6"></a>
## 6. 阈值移动 ⭐ / Threshold Shifting

最被低估的方案：**啥都不改，只调决策阈值**。模型输出的是概率，默认 0.5 变成 0/1。把阈值**降低**就能换取更高的 recall（更敢报正类）。结合 PR 曲线选最优工作点（接 5.1）。它**零成本、零泄漏风险**，常常和重采样效果相当。
The most underrated fix: **change nothing but the decision threshold.** The model outputs probabilities; 0.5 turns them into 0/1. **Lowering** the threshold buys higher recall (more willing to flag positives). Pick the best operating point from the PR curve (see 5.1). It's **zero-cost, zero-leakage**, and often matches resampling.


In [ ]:
from sklearn.metrics import precision_recall_curve

probs = clf.predict_proba(X_te)[:, 1]                  # 用基线模型的概率, 只调阈值
prec, rec, thresh = precision_recall_curve(y_te, probs)
f1s = 2*prec*rec / (prec+rec+1e-9)                     # 每个阈值对应的 F1
best_thresh = thresh[np.argmax(f1s[:-1])]             # 使 F1 最大的阈值

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(thresh, prec[:-1], label="precision"); axes[0].plot(thresh, rec[:-1], label="recall")
axes[0].plot(thresh, f1s[:-1], label="F1", lw=2)
axes[0].axvline(0.5, color="gray", ls="--", label="默认 0.5"); axes[0].axvline(best_thresh, color="red", ls="--", label=f"最优 {best_thresh:.2f}")
axes[0].set_xlabel("阈值 threshold"); axes[0].legend(); axes[0].set_title("阈值 vs 指标")
axes[1].plot(rec, prec); axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
axes[1].set_title(f"PR 曲线 (AUC={average_precision_score(y_te, probs):.3f})")
plt.tight_layout(); plt.show()
print(f"默认阈值 0.5:  F1 = {f1_score(y_te, probs>0.5):.3f}")
print(f"最优阈值 {best_thresh:.2f}: F1 = {f1_score(y_te, probs>best_thresh):.3f}")
print("仅调阈值(没碰数据/没改模型)就提升了 F1 — 零成本、零泄漏风险")


<a id="7"></a>
## 7. 横向对比 + 小结 / Showdown & Summary

用 **PR-AUC**（不平衡下最可靠、与阈值无关）横向比较各策略。一个重要结论：**重采样/加权主要改变默认阈值下的 P/R 平衡，并不必然提升底层的排序能力(PR-AUC)**——所以"调阈值"常常和"SMOTE"效果相当，但成本低得多。别迷信 SMOTE。
Compare strategies via **PR-AUC** (most reliable under imbalance, threshold-free). An important takeaway: **resampling/weighting mainly shift the P/R balance at the default threshold; they don't necessarily improve the underlying ranking (PR-AUC)** — so "threshold tuning" often matches "SMOTE" at far lower cost. Don't over-trust SMOTE.


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
strategies = {
    "基线 baseline":      LogisticRegression(max_iter=1000),
    "class_weight":       LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SMOTE(pipeline)":    ImbPipeline([("s", SMOTE(random_state=0)), ("c", LogisticRegression(max_iter=1000))]),
}
print(f"{'策略 strategy':<20} {'PR-AUC':>8} {'F1':>7} {'recall':>8}")
for name, model in strategies.items():
    pr = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="average_precision").mean()
    f1 = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="f1").mean()
    rc = cross_val_score(model, X_tr, y_tr, cv=StratifiedKFold(5), scoring="recall").mean()
    print(f"{name:<22} {pr:>8.3f} {f1:>7.3f} {rc:>8.3f}")
print("\nPR-AUC 受策略影响不大(它衡量阈值无关的排序能力);")
print("重采样/权重主要改变默认阈值下的 P/R 平衡 → 这就是为何'调阈值'常与'SMOTE'相当但成本更低")


```
准确率悖论: 1% 正类时永远猜多数=99%准但0召回 → 看 recall/precision/PR-AUC(5.1)
class_weight='balanced': 给少数类错误更高权重(代价敏感); 省事无泄漏, 首选
重采样: 欠采样(丢信息)/过采样(易过拟合)/SMOTE(近邻插值合成, 最常用)
SMOTE 泄漏 ⭐: 必须在 CV 内只对训练折做 → 用 imblearn.pipeline.Pipeline
阈值移动: 降阈值换 recall, 零成本零泄漏, 按 PR 曲线选工作点
关键认知: 重采样/权重主要移动工作点, 不必然提升 PR-AUC → 别迷信 SMOTE
```

### 💡 面试速查 / Interview cheat-sheet
1. **准确率失效** → 看 recall / precision / **PR-AUC**。
   Accuracy fails → use recall / precision / PR-AUC.
2. **class_weight='balanced'** 改损失加权，最省事无泄漏。
   class_weight='balanced' reweights the loss; easiest, no leakage.
3. **SMOTE = 近邻插值合成**少数类（非复制）。
   SMOTE = interpolated synthesis (not copying).
4. **SMOTE 必须在 CV 内**（imblearn Pipeline），否则泄漏。
   SMOTE must be inside CV (imblearn Pipeline) or it leaks.
5. **阈值移动**零成本换 recall；重采样不必然提升排序(PR-AUC)。
   Threshold shifting buys recall for free; resampling doesn't necessarily improve ranking.

### 下一节 / Next
**3.12 流水线**——把全部预处理(填补/缩放/编码/重采样)和模型打包成一个 Pipeline，一劳永逸地防泄漏、便于调参和部署。
**3.12 Pipelines** — bundle all preprocessing (impute/scale/encode/resample) with the model into one Pipeline: leakage-proof by construction, easy to tune and deploy.
